# Transcript De-identification Pipeline

This notebook demonstrates the full de-identification pipeline:
1. **Detection**: Find PII in transcript text
2. **Replacement**: Replace PII with realistic fake data
3. **Output**: Generate de-identified transcript and re-identification mapping

In [ ]:
# Setup path for imports
import sys
from pathlib import Path

# Add this directory to path
sys.path.insert(0, str(Path('.').resolve()))

## 1. PII Detection

The detection module uses spaCy NER with custom patterns optimized for ASR transcripts.

In [ ]:
from detection import PiiDetector
from detection.settings import DetectorSettings

# Create detector with default settings
detector = PiiDetector()

# Or with custom settings
# settings = DetectorSettings(model_name="en_core_web_sm")  # smaller model
# detector = PiiDetector(settings=settings)

In [ ]:
# Sample transcript text
sample_text = """
Hi Maria, how are you today?
I'm good, thanks! I go to Lincoln High School.
That's great. I'm Austin, your tutor for today.
Nice to meet you, Austin. I'm 15 years old.
Perfect. Let's start with some math problems.
""".strip()

print("Input text:")
print(sample_text)

In [ ]:
# Detect PII
ner_result = detector.detect(sample_text)

print("\n=== Detection Results ===")
print(f"Total distinct PII items: {ner_result.distinct_pii.count()}")
print(f"Total occurrences: {len(ner_result.pii_occurrences)}")

print("\n--- Distinct PII by Category ---")
for pii_type in ["NAME", "LOCATION", "SCHOOL", "DATE", "AGE", "PHONE", "EMAIL", "URL", "MISC_ID"]:
    items = ner_result.distinct_pii.get(pii_type)
    if items:
        print(f"{pii_type}: {items}")

In [ ]:
# Show occurrence details
print("\n--- PII Occurrences ---")
for occ in ner_result.pii_occurrences:
    print(f"  [{occ.pii_type}] '{occ.text}' at positions {occ.start}-{occ.end}")

## 2. PII Replacement

The replacement module generates realistic fake data to replace detected PII.

In [ ]:
from replacement import TranscriptDeidentifier, create_actions_config

# Configure actions for each PII type
actions = create_actions_config(
    default_action="replace",  # Replace with fake data
    # redact=["EMAIL", "PHONE"]  # Uncomment to redact these instead
)

print("Actions configuration:")
for pii_type, action in actions.items():
    print(f"  {pii_type}: {action}")

In [ ]:
# Create transcript dict (simulating loaded JSON)
transcript = {
    "utterances": [
        {"text": line, "speaker": "Tutor" if i % 2 == 0 else "Student"}
        for i, line in enumerate(sample_text.split("\n"))
        if line.strip()
    ]
}

print("Original transcript:")
for utt in transcript["utterances"]:
    print(f"  [{utt['speaker']}] {utt['text']}")

In [ ]:
# De-identify the transcript
deidentifier = TranscriptDeidentifier(seed=42)  # Use seed for reproducibility
deid_transcript, reid_dict = deidentifier.deidentify(ner_result, transcript, actions)

print("\n=== De-identified Transcript ===")
for utt in deid_transcript["utterances"]:
    print(f"  [{utt['speaker']}] {utt['text']}")

In [ ]:
# Show re-identification mapping
print("\n=== Re-identification Mapping ===")
print("(Replacement → Original)")
for replacement, original in reid_dict.items():
    print(f"  '{replacement}' → '{original}'")

## 3. Redaction Mode

Instead of replacing with fake data, you can redact with placeholder labels.

In [ ]:
# Configure for redaction
redact_actions = create_actions_config(default_action="redact")

# Re-create transcript (since it was modified in place)
transcript_for_redaction = {
    "utterances": [
        {"text": line, "speaker": "Tutor" if i % 2 == 0 else "Student"}
        for i, line in enumerate(sample_text.split("\n"))
        if line.strip()
    ]
}

# De-identify with redaction
redact_deidentifier = TranscriptDeidentifier(seed=42)
redacted_transcript, _ = redact_deidentifier.deidentify(
    ner_result, transcript_for_redaction, redact_actions
)

print("=== Redacted Transcript ===")
for utt in redacted_transcript["utterances"]:
    print(f"  [{utt['speaker']}] {utt['text']}")

## 4. Working with Files

Example of processing transcript files.

In [ ]:
import json

def process_transcript_file(input_path: str, output_path: str, seed: int = 42):
    """
    Process a transcript file through the de-identification pipeline.
    
    Args:
        input_path: Path to input transcript JSON
        output_path: Path to output de-identified JSON
        seed: Random seed for reproducibility
    """
    # Load transcript
    with open(input_path, "r", encoding="utf-8") as f:
        transcript = json.load(f)
    
    # Convert to text
    text = "\n".join(
        utt.get("text", "") 
        for utt in transcript.get("utterances", [])
    )
    
    # Detect PII
    detector = PiiDetector()
    ner_result = detector.detect(text)
    
    # Configure actions
    actions = create_actions_config(default_action="replace")
    
    # De-identify
    deidentifier = TranscriptDeidentifier(seed=seed)
    deid_transcript, reid_dict = deidentifier.deidentify(
        ner_result, transcript, actions
    )
    
    # Save outputs
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(deid_transcript, f, indent=2)
    
    reid_path = output_path.replace(".json", "_reid.json")
    with open(reid_path, "w", encoding="utf-8") as f:
        json.dump(reid_dict, f, indent=2)
    
    return deid_transcript, reid_dict

# Example usage (uncomment to run):
# deid, reid = process_transcript_file("input.json", "output_deid.json")

## 5. Name Replacement Details

The name replacer preserves characteristics of the original names.

In [ ]:
from replacement.name_replacer import NameReplacer, detect_gender, detect_name_type

# Test name detection
test_names = ["Maria", "John", "Smith", "Williams", "Austin"]

print("Name Analysis:")
for name in test_names:
    name_type = detect_name_type(name)
    gender = detect_gender(name) if name_type == "first" else "N/A"
    print(f"  {name}: type={name_type}, gender={gender}")

In [ ]:
# Demonstrate name replacement
name_replacer = NameReplacer(seed=42)

print("\nName Replacements (preserving initial):")
for name in ["Maria", "John", "Austin"]:
    replacement = name_replacer.replace_name_auto(name)
    print(f"  {name} → {replacement}")

## 6. Statistics Summary

In [ ]:
print("=== Pipeline Statistics ===")
print(f"\nInput text length: {len(sample_text)} characters")
print(f"Number of utterances: {len(transcript['utterances'])}")
print(f"\nDetected PII:")
print(f"  - Distinct items: {ner_result.distinct_pii.count()}")
print(f"  - Total occurrences: {len(ner_result.pii_occurrences)}")
print(f"\nReplacements made: {len(reid_dict)}")